In [1]:
import re
import sqlite3
import pandas as pd

In [2]:
conn = sqlite3.connect('lectures_FS2026.db')
c_lect = conn.cursor()

In [3]:
# lecture database entry

# c.execute('''
# CREATE TABLE IF NOT EXISTS lectures (
#   id INTEGER PRIMARY KEY,
#   number TEXT,
#   title TEXT,
#   url TEXT,
#   type TEXT,
#   ects TEXT,
#   hours TEXT,
#   abstract TEXT,
#   learning_objective TEXT,
#   content TEXT,
#   lecture_notes TEXT,
#   literature TEXT
# )
# ''')

In [4]:
def search_lectures(keywords):
    # keywords: list of strings
    fields = ["title", "abstract", "learning_objective", "content", "lecture_notes", "literature"]
    query = "SELECT * FROM lectures WHERE "
    conditions = []
    params = []
    for kw in keywords:
        subconds = [f"{field} LIKE ?" for field in fields]
        conditions.append("(" + " OR ".join(subconds) + ")")
        params.extend([f"%{kw}%"] * len(fields))
    query += " AND ".join(conditions)
    c_lect.execute(query, params)
    results = c_lect.fetchall()
    return results

In [6]:
# results = search_lectures(["fluidic","micro"])
results = search_lectures(["biocompatible","material"])

# clean duplicates
seen = set()
unique_results = []
for row in results:
    identifier = (row[1], row[2])  # Use (number, title) as a unique identifier
    if identifier not in seen:
        seen.add(identifier)
        unique_results.append(row)
results = unique_results

# only show title and url
for row in results:
    # print(f"ID: {row[0]}")
    print(f"{row[1]}: {row[2]}")
    # print(f"URL: https://{row[3]}")
    print(f"URL: {row[3]}")
    print("-" * 40)

376-1614-00L: Principles in Tissue Engineering
URL: https://www.vvz.ethz.ch/Vorlesungsverzeichnis/lerneinheit.view?semkez=2026S&ansicht=ALLE&lerneinheitId=198786&lang=en
----------------------------------------
376-1624-00L: Practical Methods in Biofabrication
URL: https://www.vvz.ethz.ch/Vorlesungsverzeichnis/lerneinheit.view?semkez=2026S&ansicht=ALLE&lerneinheitId=198146&lang=en
----------------------------------------
376-1714-AAL: Biocompatible Materials
URL: https://www.vvz.ethz.ch/Vorlesungsverzeichnis/lerneinheit.view?semkez=2026S&ansicht=ALLE&lerneinheitId=199964&lang=en
----------------------------------------


### Check for duplicates

Lectures

In [7]:
# check if there are any duplicate lectures in the database
c_lect.execute('''
    SELECT number, title, COUNT(*) c FROM lectures GROUP BY number HAVING c > 1
''')
duplicates = c_lect.fetchall()
if duplicates:
    print("Duplicate lectures found:")
    for dup in duplicates:
        print(f"Count: {dup[2]} - {dup[0]}: {dup[1]}")
else:
    print("No duplicate lectures found.")

No duplicate lectures found.


Lecturers

In [8]:
# check if there are any duplicate lecturers in the database
c_lect.execute('''
    SELECT name, COUNT(*) c FROM lecturers GROUP BY name HAVING c > 1
''')
duplicates = c_lect.fetchall()
if duplicates:
    print("Duplicate lecturers found:")
    for dup in duplicates:
        print(f"Lecture Number: {dup[0]}, Count: {dup[1]}")
else:
    print("No duplicate lectures found.")

No duplicate lectures found.


## Find nearest neighbors using embeddings

In [9]:
import sqlite_vec

In [10]:
# Connect to VVZ SQLite database
# conn = sqlite3.connect('lectures.db')
# c_lect = conn.cursor()

# Connect to embedding database (or create it if it doesn't exist)
conn_emb = sqlite3.connect('embeddings_FS2026.db')
c_emb = conn_emb.cursor()

# load the sqlite vector extension
conn_emb.enable_load_extension(True)
sqlite_vec.load(conn_emb)
conn_emb.enable_load_extension(False)

vec_version, = conn_emb.execute("select vec_version()").fetchone()
print(f"vec_version={vec_version}")

vec_version=v0.1.6


In [11]:
from sentence_transformers import SentenceTransformer
import numpy as np

# Load a pre-trained model and generate embeddings
model = SentenceTransformer("all-MiniLM-L6-v2")

In [12]:
# Define the target embedding (e.g., from a query or another source)
# target_embedding = np.random.rand(384).astype('float32')

# lecture_number = "376-1714-00L"
lecture_number = "376-1614-00L"

# get lecture data from lecture number
c_lect.execute('SELECT number, title, abstract, learning_objective, content, lecture_notes, literature FROM lectures WHERE number=?', (lecture_number,))
lecture = c_lect.fetchone()

texts = []

number, title, abstract, learning_objective, content, lecture_notes, literature = lecture

for text in [title, abstract, learning_objective, content, lecture_notes, literature]:
    if text:
        texts.append(text)

custom_text_additions = [
    # "Machine learning in biology"
    # "Advanced Topics in Control"
]

texts.extend(custom_text_additions)

target_embeddings = model.encode(texts)

# Perform a similarity search in the virtual table for each text embedding
results_list = []
for target_embedding in target_embeddings:
    query = '''
    SELECT lecture_number, distance
    FROM vss_embeddings
    WHERE embedding MATCH ?
    AND k = 10
    '''

    # Execute the query with the target embedding
    results = c_emb.execute(query, (target_embedding,)).fetchall()
    results_list.append(results)

# Flatten results and remove duplicates
# for entries with same lecture number, keep the one with highest similarity (lowest distance)
results = list(set([item for sublist in results_list for item in sublist]))
results.sort(key=lambda x: x[1])  # Sort by distance (similarity)
seen_lectures = set()
unique_results = []
for lecture_number, similarity in results:
    if lecture_number not in seen_lectures:
        seen_lectures.add(lecture_number)
        unique_results.append((lecture_number, similarity))
results = unique_results

# print lecture data and similarity from results
for lecture_number, similarity in results:
    c_lect.execute('SELECT number, title, url, abstract, learning_objective, content, lecture_notes, literature FROM lectures WHERE number=?', (lecture_number,))
    lecture = c_lect.fetchone()
    number, title, url, abstract, learning_objective, content, lecture_notes, literature = lecture
    print(f"{number}: {title}, https://{url}, Similarity: {similarity}")

376-1614-00L: Principles in Tissue Engineering, https://https://www.vvz.ethz.ch/Vorlesungsverzeichnis/lerneinheit.view?semkez=2026S&ansicht=ALLE&lerneinheitId=198786&lang=en, Similarity: 0.0
701-0034-06L: Integrated Practical: Soil, https://https://www.vvz.ethz.ch/Vorlesungsverzeichnis/lerneinheit.view?semkez=2026S&ansicht=ALLE&lerneinheitId=199319&lang=en, Similarity: 0.6299595832824707
327-4105-00L: Integrity of Materials and Structures, https://https://www.vvz.ethz.ch/Vorlesungsverzeichnis/lerneinheit.view?semkez=2026S&ansicht=ALLE&lerneinheitId=198262&lang=en, Similarity: 0.6353561282157898
151-0946-00L: Macromolecular Engineering: Networks and Gels, https://https://www.vvz.ethz.ch/Vorlesungsverzeichnis/lerneinheit.view?semkez=2026S&ansicht=ALLE&lerneinheitId=198926&lang=en, Similarity: 0.65692138671875
636-0115-00L: Biotechnology of Microbes, https://https://www.vvz.ethz.ch/Vorlesungsverzeichnis/lerneinheit.view?semkez=2026S&ansicht=ALLE&lerneinheitId=199563&lang=en, Similarity: 0

## Close connection

In [13]:
# close the connections
conn.close()
conn_emb.close()